# CALPHAD workflow with ATAT and machine-learned potentials

This tutorial fits CALPHAD databases with the `CalphadMaker` workflow and
draws the phase diagrams. It covers three binaries that Zhu et al. (npj Comput.
Mater. 11, 340 (2025)) also computed: Ni-Re, Co-Ni and Cr-V. The energies come
from four machine-learned potentials, so no DFT is needed. The workflow is
described in the [force field documentation](https://materialsproject.github.io/atomate2/user/codes/forcefields.html#calphad).

This workflow is new and has not been tested widely. It might still change in
future versions.

Contents:

- Recommendations
- How the workflow works
- Installation and the potentials
- 1. Ni-Re, with every step of the workflow
- 2. Co-Ni
- 3. Cr-V
- 4. Choosing the liquid settings
- 5. Comparison with PhaseForge
- Known limitations

## Recommendations

These settings worked for the three binaries with all four potentials.

- Use the default liquid cell of 864 atoms. With 256 atoms the liquid mixing
  energy of one potential was off by 2 kJ/mol.
- Set `melt_temperature` high enough that every composition melts. We used
  4500 K for Ni-Re, 3000 K for Co-Ni and 3500 K for Cr-V.
- Run all compositions at one `liquid_temperature`, as low as possible while
  every composition stays liquid. We used 2800 K for Ni-Re, 2000 K for Co-Ni
  and 2400 K for Cr-V.
- Keep the phonon step, which is on by default. With three of the four
  potentials, the vibrational entropy of the solids turned the Cr-V liquidus
  maximum into a minimum, as in experiment.
- Check that `relaxation_strain` of each solid is below 0.1, that
  `imaginary_fraction` of each solid is close to 0 and that
  `mean_squared_displacement` of each liquid is well above 1 Å².
- One liquid run per SQS was enough for these binaries. Between runs with
  different random seeds, the liquidus maximum or minimum of Co-Ni and Cr-V
  moved by 11 to 23 K (one standard deviation). `energy_standard_error` shows
  this scatter from a single run. Average more runs if it is too large.

## How the workflow works

The workflow follows the `sqs2tdb` approach of ATAT (van de Walle et al.,
Calphad 58, 70 (2017)). It has five steps.

1. `sqs2tdb` copies the special quasirandom structures (SQS) of each lattice
   from the ATAT database. An SQS is a small cell whose random occupation of
   the sites mimics a random alloy. At level 2 these are the pure elements and
   the compositions with 25, 50 and 75 at% of each element.
2. Each solid SQS is relaxed with the force field, including the cell.
3. The phonons of each relaxed solid SQS give its harmonic vibrational entropy.
   The atoms are relaxed again at fixed cell, and the displacements are run
   in a diagonal supercell with all lattice vectors at least 20 Å long. The
   entropy is phonopy's sum over a q-point mesh at 3000 K.
4. Each liquid SQS is repeated three times along each lattice vector, which
   gives 864 atoms. It is melted at `melt_temperature` and then run at
   `liquid_temperature`. Both runs are NPT at zero pressure. The mean potential
   energy of the second run is the energy of the liquid.
5. `sqs2tdb` fits the mixing energies and vibrational entropies of each
   lattice and writes one TDB file. A TDB file is a CALPHAD database that
   pycalphad can read.

The mixing energy of a phase of A and B is a Redlich-Kister sum,
x<sub>A</sub> x<sub>B</sub> [L0 + L1 (x<sub>A</sub> − x<sub>B</sub>) + ...].
The default terms fit L0 and L1. For a solid, each term is a + b T. The
mixing energy gives a, and the excess vibrational entropy gives b. The liquid
terms have no T part. The excess entropy of a solid is the vibrational entropy
of its SQS minus the mean of the end members, weighted by composition. The
entropy at 3000 K is used at all temperatures. At this temperature the excess
is close to its high temperature limit. For lattices in the SGTE database, `sqs2tdb`
takes the free energies of the pure elements from SGTE (Dinsdale, Calphad 15,
317 (1991)). The melting points of the pure elements are therefore the
experimental ones.

## Installation and the potentials

The `ase` and `phonons` extras are needed, plus MACE and pycalphad:

```
pip install 'atomate2[ase,phonons]' 'mace-torch>=0.3.16' pycalphad
```

GRACE needs `tensorpotential`, which brings TensorFlow. PyTorch and
TensorFlow each install their own CUDA libraries, so we ran GRACE in a separate
environment:

```
pip install 'atomate2[ase,phonons]' tensorpotential pycalphad
export TF_USE_LEGACY_KERAS=1
```

GRACE saves its models in `~/.cache/grace`. Set `GRACE_CACHE` to save them
somewhere else.

ATAT is not a Python package. Besides the `sqs2tdb` script, the workflow needs
three ATAT programs, which build in a few seconds:

```
curl -sSLO https://axelvandewalle.github.io/www-avdw/atat/atat3_52.tar.gz
tar xzf atat3_52.tar.gz
make -C atat/src cellcvrt nntouch lsfit
echo "set atatdir=$PWD/atat" > ~/.atat.rc
export PATH=$PWD/atat/src:$PATH
```

`sqs2tdb` calls the other programs by name, so the ATAT `src` folder must be on
the `PATH` of the shell that starts Jupyter. The next cell checks this.

In [ ]:
import shutil

for program in ("sqs2tdb", "cellcvrt", "nntouch", "lsfit"):
    if shutil.which(program) is None:
        raise OSError(f"{program} is not on the PATH")

The tutorial uses four foundation models.

- **MACE-OMAT-0-medium.** `mace_mp(model="medium-omat-0")` downloads it once and
  caches it under `~/.cache/mace`.
- **MACE-MATPES-PBE-0** and **MACE-MATPES-r2SCAN-0.** These are MACE-OMAT-0
  fine-tuned on the MatPES data set at the PBE and r2SCAN levels. Download the
  two files from the `mace_matpes_0` release of
  [mace-foundations](https://github.com/ACEsuit/mace-foundations/releases/tag/mace_matpes_0):
  - [MACE-matpes-pbe-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-pbe-omat-ft.model)
  - [MACE-matpes-r2scan-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-r2scan-omat-ft.model)
- **GRACE-2L-OMAT.** A two-layer GRACE model fitted on OMat24. `grace_fm` of
  `tensorpotential` downloads it once by name. atomate2 has no name for GRACE,
  so the calculator is given as a dictionary that points to `grace_fm`.

Set the paths to the two MACE files in the next cell. The liquid MD needs a
GPU, so MACE runs on `"cuda"`. The mixing energies are small differences of
large total energies, so MACE runs in `float64`. GRACE runs in `float64` by
default.

In [ ]:
MACE = {"device": "cuda", "default_dtype": "float64"}
GRACE = {
    "@module": "tensorpotential.calculator.foundation_models",
    "@callable": "grace_fm",
}

# Replace the two paths with the files you downloaded.
FORCE_FIELDS = {
    "MACE-OMAT-0-medium": ("MACE-MP-0", {"model": "medium-omat-0", **MACE}),
    "MACE-MATPES-PBE-0": (
        "MACE-MP-0",
        {"model": "/path/to/MACE-matpes-pbe-omat-ft.model", **MACE},
    ),
    "MACE-MATPES-r2SCAN-0": (
        "MACE-MP-0",
        {"model": "/path/to/MACE-matpes-r2scan-omat-ft.model", **MACE},
    ),
    "GRACE-2L-OMAT": (GRACE, {"model": "GRACE-2L-OMAT"}),
}

## 1. Ni-Re

This section goes through every step of the workflow for Ni-Re.

### Lattices and terms

Ni is stable in FCC_A1 and Re in HCP_A3. We fit these two lattices and the
liquid. We also fit two ordered lattices of the ATAT database, NI3SN_D019 and
NI4MO_D1A. They test whether a force field makes an ordered phase stable.

`terms` gives the lines of the `sqs2tdb` `terms.in` file. Each line is an
`order,level` pair. Order 1 gives the end members and order 2 the mixing.
Level 0 fits L0 only. The ordered lattices have two sublattices, so their lines
have one pair per sublattice. `sqs2tdb` applies each line to every
sublattice, so `2,0:1,0` fits mixing on each sublattice while the other one
is pure.

The default `2,1` fits L0 and L1. At level 2 the ordered lattices have too few
SQS for L1, so they take L0 only. We tried L1 for FCC_A1, HCP_A3 and LIQUID
with L0 for the ordered lattices. NI3SN_D019 was then stable up to about
3000 K with MACE-OMAT-0-medium and GRACE-2L-OMAT. So this tutorial fits L0
only for every lattice. Adding L1 does not change L0, since the SQS
compositions are symmetric around 50 at%.

### Liquid temperatures

The liquid temperatures depend on the system, so they have no defaults. The
melt at 4500 K makes every composition melt within the 10 ps of the first run.
All compositions are then run at 2800 K. This is below the melting point of Re
(3458 K in SGTE). The melted cells still stay liquid during the 20 ps run. The
section on the liquid settings explains this choice.

In [ ]:
from atomate2.forcefields.flows.calphad import CalphadMaker

TERMS = {
    "FCC_A1": ["1,0", "2,0"],
    "HCP_A3": ["1,0", "2,0"],
    "NI3SN_D019": ["1,0:1,0", "2,0:1,0"],
    "NI4MO_D1A": ["1,0:1,0", "2,0:1,0"],
    "LIQUID": ["1,0", "2,0"],
}


def get_maker(model: str, **kwargs) -> CalphadMaker:
    """Return a CALPHAD maker with one of the potentials."""
    force_field, calculator_kwargs = FORCE_FIELDS[model]
    return CalphadMaker.from_force_field_name(
        force_field, calculator_kwargs=calculator_kwargs, **kwargs
    )


NI_RE = {
    "melt_temperature": 4500,
    "liquid_temperature": 2800,
    "lattices": list(TERMS),
    "terms": TERMS,
}
flow = get_maker("MACE-OMAT-0-medium", **NI_RE).make(["Ni", "Re"])
[job.name for job in flow.jobs]

### Running the workflow

The flow computes 29 SQS, 24 solids and 5 liquids, plus the phonons of each
solid. On one A100 GPU the solid relaxations took a few minutes together. Each liquid run took 40 to 46 minutes
with MACE and about 22 minutes with GRACE-2L-OMAT. `run_locally` runs the
liquids one after the other, which takes about 3.5 hours with MACE. With a
workflow manager the five liquids can run at the same time on five GPUs. The
next cell is therefore off by default.

`create_folders=True` gives each job its own folder. The fit job runs
`sqs2tdb` in its folder.

In [ ]:
from jobflow import run_locally

RUN_WORKFLOW = False
if RUN_WORKFLOW:
    responses = run_locally(flow, create_folders=True, ensure_success=True)
    calphad_doc = responses[flow.output.uuid][1].output

### Checking the output

The output is a `CalphadDoc`. It holds the TDB file in `tdb` and the energy of
each SQS in `calculations`. Each solid also has its `vibrational_entropy` in
k_B per SQS cell. There are four checks.

- `relaxation_strain` of a solid measures how far the relaxed cell is from the
  ideal lattice. The ATAT `checkrelax` help calls values above 0.1 too large.
- `is_force_converged` says whether the relaxation reached `fmax` within its
  steps.
- `imaginary_fraction` of a solid is the part of its phonon frequencies on the
  q-point mesh that are imaginary. If it is above `max_imaginary_fraction`,
  0.03 by default, for one SQS of a lattice, that lattice is fitted to the
  energies only and the fit job gives a warning.
- `mean_squared_displacement` of a liquid keeps growing during the run, because
  the atoms keep moving. In a crystal it stays well below 1 Å².

In [ ]:
import pandas as pd

checks = None
if RUN_WORKFLOW:
    checks = pd.DataFrame(
        [
            calc.model_dump(exclude={"structure", "dir_name"})
            for calc in calphad_doc.calculations
        ]
    )
checks

The next table gives these checks from the output of our first run at 2800 K
with each potential. The solids were relaxed with `fmax` = 0.001 eV/Å and at
most 500 steps. The default is now 0.01 eV/Å and 1000 steps. The last five
columns are the mean squared displacement of the liquid in Å², by at% Re.

| Potential | Largest strain | Not converged | 0 | 25 | 50 | 75 | 100 |
|---|---|---|---|---|---|---|---|
| MACE-OMAT-0-medium | 0.033 | 2 of 24 | 118.0 | 76.6 | 47.2 | 31.2 | 16.8 |
| MACE-MATPES-PBE-0 | 0.028 | 1 of 24 | 119.1 | 75.1 | 45.5 | 28.8 | 16.1 |
| MACE-MATPES-r2SCAN-0 | 0.026 | 3 of 24 | 117.0 | 66.1 | 41.3 | 24.5 | 14.3 |
| GRACE-2L-OMAT | 0.038 | 4 of 24 | 109.3 | 74.5 | 49.6 | 27.4 | 15.3 |

We computed the vibrational entropies after these runs, from the relaxed SQS
and with the same phonon settings as the workflow. No solid SQS had an
imaginary frequency on the mesh. All strains are far below 0.1. We repeated the MACE relaxations with 2000
steps, and the energies did not change. We did not repeat the GRACE
relaxations. Every liquid has a mean squared displacement far above 1 Å², so
every liquid SQS has melted.

### Fitting the TDB and plotting

To draw the diagrams without running the workflow, we refit the TDB files from
the results of our runs. The files hold the total energy in eV of each SQS
cell, and the vibrational entropy in k_B of each solid SQS cell. The liquid
energies are per 32-atom SQS, from the first of our MD runs.

`fit_tdb` is the last job of the flow, and `.original` calls its function
directly. It writes the `sqs2tdb` folders into the current folder, so each fit
runs in a folder of its own.

In [ ]:
import contextlib
import json
from pathlib import Path

from mock_vasp import TEST_DIR

from atomate2.common.jobs.calphad import fit_tdb


def fit_tdbs(system: str, elements: list[str], terms: dict) -> dict[str, str]:
    """Fit the TDB file of each potential from the energies of our runs."""
    path = TEST_DIR / "common" / "calphad" / f"{system}_energies.json"
    tdbs = {}
    for model, calculations in json.loads(path.read_text()).items():
        fit_dir = Path("calphad_fits", system, model)
        fit_dir.mkdir(parents=True, exist_ok=True)
        with contextlib.chdir(fit_dir):
            tdbs[model] = fit_tdb.original(elements, 2, terms, calculations).tdb
    return tdbs


tdbs = fit_tdbs("ni_re", ["Ni", "Re"], TERMS)

The next cell draws the four diagrams. The dotted line is the experimental
peritectic at 1834 K.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from pycalphad import Database, binplot
from pycalphad import variables as v

# one colour per phase in every diagram, away from the green tie lines and the
# red three-phase lines that pycalphad draws
PHASE_COLORS = {
    "LIQUID": "#F4C800",
    "FCC_A1": "#00538A",
    "BCC_A2": "#704AA4",
    "HCP_A3": "#FF8E00",
    "NI3SN_D019": "#7BD1EC",
    "NI4MO_D1A": "#803E75",
}


def phase_legend(phases: list[str]) -> tuple[list, dict[str, str]]:
    """Return the legend handles and the colour of each phase for binplot.

    pycalphad names a second composition set of a phase as e.g. NI4MO_D1A#2.
    """
    colors = {phase: PHASE_COLORS[phase.split("#")[0]] for phase in phases}
    names = sorted({phase.split("#")[0] for phase in phases})
    handles = [mpatches.Patch(color=PHASE_COLORS[p], label=p) for p in names]
    return handles, colors


def plot_diagrams(
    tdbs: dict[str, str], elements: list[str], temperatures: tuple, line: float = 0
) -> None:
    """Draw the phase diagram of each potential, with a dotted line at line K."""
    species = [element.upper() for element in elements]
    conditions = {v.X(species[1]): (0, 1, 0.02), v.T: temperatures, v.P: 101325, v.N: 1}
    fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
    for ax, (model, tdb) in zip(axes.flat, tdbs.items(), strict=True):
        db = Database.from_string(tdb, fmt="tdb")
        plot_kwargs = {"ax": ax, "legend_generator": phase_legend}
        binplot(db, species, sorted(db.phases), conditions, plot_kwargs=plot_kwargs)
        if line:
            ax.axhline(line, color="black", ls=":")
        ax.get_legend().remove()
        ax.set_title(model)
        ax.label_outer()
    fig.legend(handles=phase_legend(sorted(db.phases))[0], loc="center right")
    fig.tight_layout(rect=(0, 0, 0.85, 1))
    plt.show()


plot_diagrams(tdbs, ["Ni", "Re"], (1000, 4000, 25), line=1834)

![Ni-Re phase diagrams with the four potentials](calphad_workflow_figures/ni_re_phase_diagrams.png)

The next cell reads L0 = a + b T from each TDB file, with a in J/mol and b in
J/(mol K).

In [ ]:
import re

# a TDB parameter such as +1357.5-1.9061*T, or -3860.4 for the liquid
TERM = re.compile(r"([-+][\d.]+)(?:([-+][\d.]+)\*T)?")


def get_mixing_terms(
    tdbs: dict[str, str], lattices: list[str], level: int
) -> pd.DataFrame:
    """Return L0 to L<level> = a + b T of each lattice, read from the TDB files.

    a is in J/mol and b in J/(mol K). b is 0 for the liquid.
    """
    rows = {}
    for model, tdb in tdbs.items():
        rows[model] = {}
        for lattice in lattices:
            for order in range(level + 1):
                line = next(
                    line
                    for line in tdb.splitlines()
                    if f"L({lattice}," in line and f";{order})" in line
                )
                a, b = TERM.fullmatch(line.split()[3]).groups()
                rows[model][f"{lattice} L{order} a"] = float(a)
                rows[model][f"{lattice} L{order} b"] = float(b or 0)
    return pd.DataFrame(rows).T.round(2)


get_mixing_terms(tdbs, ["FCC_A1", "HCP_A3", "LIQUID"], 0)

With our results the cell gives these values of L0 in J/mol.

| Potential | FCC_A1 | HCP_A3 | LIQUID |
|---|---|---|---|
| MACE-OMAT-0-medium | +7168 + 0.41 T | +15938 − 0.36 T | +366 |
| MACE-MATPES-PBE-0 | −2723 + 2.85 T | +2481 + 2.89 T | −17579 |
| MACE-MATPES-r2SCAN-0 | +3980 − 0.12 T | +14100 − 0.03 T | −13920 |
| GRACE-2L-OMAT | +1358 − 1.91 T | +892 + 3.36 T | −3860 |

A b of 1 J/(mol K) is an excess vibrational entropy of −0.03 k_B/atom at
50 at% Re. So the vibrational entropy changes FCC_A1 and HCP_A3 little. It
changes the ordered lattices more. Their b values reach 56 J/(mol K) in
size.

### Phase diagrams

The four potentials give different diagrams on the Ni side.

- **MACE-OMAT-0-medium.** At about 1890 K the liquid and HCP_A3 form
  NI4MO_D1A in a peritectic. Near Ni the liquid ends at about 1715 K, in a
  eutectic with FCC_A1 and NI4MO_D1A.
- **MACE-MATPES-PBE-0 and MACE-MATPES-r2SCAN-0.** The liquid is stable far
  below the melting point of Ni. It ends near 30 at% Re at about 1585 K (PBE)
  and 1475 K (r2SCAN). There it freezes into NI4MO_D1A and, with PBE, also
  HCP_A3.
- **GRACE-2L-OMAT.** At about 2150 K the liquid and HCP_A3 form NI4MO_D1A in a
  peritectic. Near Ni, FCC_A1 forms from the liquid and NI4MO_D1A at about
  1745 K.

With the two MatPES potentials, L0 of the liquid is much more negative than L0
of FCC_A1. Adding Re to Ni therefore stabilizes the liquid and lowers the
liquidus. With MACE-OMAT-0-medium and GRACE-2L-OMAT the two are closer, so the
liquidus stays near the melting point of Ni.

### Comparison with experiment

In experiment Ni-Re has a peritectic L + HCP → FCC at 1834 K (1561 °C). The
liquid at the peritectic has about 11 at% Re, FCC 20 at% and HCP 60 at%
(Boettinger et al., Metall. Mater. Trans. A (2019)). No ordered phase has
been reported in Ni-Re as far as we know.

MACE-OMAT-0-medium comes closest. Its peritectic is about 55 K above the
experimental one. But it forms the ordered NI4MO_D1A phase in place of
FCC_A1. The peritectic of GRACE-2L-OMAT is about 315 K too high. Without the
vibrational entropy it was about 40 K too low. So here the vibrational
entropy moves the diagram away from experiment. It makes NI4MO_D1A more
stable and HCP_A3 less stable at high temperature.

All four potentials make ordered phases stable. NI4MO_D1A is stable up to
1890 K (MACE-OMAT-0-medium), 1670 K (PBE), 1485 K (r2SCAN) and 2150 K
(GRACE-2L-OMAT). With PBE, NI3SN_D019 is stable up to 1560 K. These
temperatures are from `pycalphad.equilibrium` on a grid with steps of 0.02 in
composition and 5 K in temperature. The known limitations at the end discuss
why the ordered lattices may be too stable.

## 2. Co-Ni

Co-Ni uses the same potentials and steps as Ni-Re. Ni is stable in FCC_A1 and
Co in HCP_A3, so we fit these two lattices and the liquid with the default
terms, L0 and L1. The liquid is melted at 3000 K and run at 2000 K. This is
about 230 K above the melting points of both elements. The solids were relaxed
with the default settings.

In [ ]:
CO_NI = {
    "melt_temperature": 3000,
    "liquid_temperature": 2000,
    "lattices": ["FCC_A1", "HCP_A3", "LIQUID"],
}
maker = get_maker("MACE-OMAT-0-medium", **CO_NI)
flow = maker.make(["Co", "Ni"])
co_ni_terms = {lattice: maker.terms[lattice] for lattice in maker.lattices}

tdbs = fit_tdbs("co_ni", ["Co", "Ni"], co_ni_terms)
plot_diagrams(tdbs, ["Co", "Ni"], (300, 2200, 10))
get_mixing_terms(tdbs, maker.lattices, 1)

![Co-Ni phase diagrams with the four potentials](calphad_workflow_figures/co_ni_phase_diagrams.png)

Near the melting points the liquid and FCC_A1 are close in free energy, so
the two-phase region is only a few K wide. With GRACE-2L-OMAT it is so narrow
that pycalphad's mapping in `binplot` loses it between about 15 and 87 at% Ni.
The next cell finds the liquidus and the solidus with `pycalphad.equilibrium`
on a grid instead and draws them for all four potentials.

In [ ]:
from pycalphad import equilibrium


def plot_melting(
    tdbs: dict[str, str], elements: list[str], temperatures: tuple
) -> None:
    """Draw the liquidus (solid line) and the solidus (dashed) of each potential.

    The liquidus is the lowest temperature of the grid at which only the liquid
    is stable. The solidus is the highest temperature without liquid.
    """
    species = [element.upper() for element in elements]
    conditions = {
        v.X(species[1]): (0, 1.001, 0.02),
        v.T: temperatures,
        v.P: 101325,
        v.N: 1,
    }
    _, ax = plt.subplots(figsize=(7, 5))
    for model, tdb in tdbs.items():
        db = Database.from_string(tdb, fmt="tdb")
        eq = equilibrium(db, [*species, "VA"], sorted(db.phases), conditions)
        phases = eq.Phase.values.squeeze()
        liquid = (phases == "LIQUID").any(axis=-1)
        solid = ((phases != "LIQUID") & (phases != "")).any(axis=-1)
        temperature = eq.T.values
        liquidus = [temperature[~column].min() for column in solid.T]
        solidus = [temperature[~column].max() for column in liquid.T]
        x = eq[f"X_{species[1]}"].values
        (line,) = ax.plot(x, liquidus, label=model)
        ax.plot(x, solidus, ls="--", color=line.get_color())
    ax.set_xlabel(f"X({species[1]})")
    ax.set_ylabel("Temperature (K)")
    ax.legend()
    plt.show()


plot_melting(tdbs, ["Co", "Ni"], (1600, 1800, 2))

![Liquidus and solidus of Co-Ni with the four potentials](calphad_workflow_figures/co_ni_melting.png)

In our runs the largest `relaxation_strain` was 0.011. One or two of the 10
solid relaxations did not reach `fmax` within 1000 steps. No solid SQS had an
imaginary phonon frequency on the mesh. The mean squared displacement of the
liquids at 2000 K was 36 to 58 Å². The cell gives these mixing terms. The
first number is a in J/mol and the second b in J/(mol K).

| Potential | FCC_A1 L0 | FCC_A1 L1 | HCP_A3 L0 | HCP_A3 L1 | LIQUID L0 | LIQUID L1 |
|---|---|---|---|---|---|---|
| MACE-OMAT-0-medium | −3268, +3.29 | −914, +0.52 | −3761, +2.64 | −1351, +0.49 | +677 | −922 |
| MACE-MATPES-PBE-0 | −6690, +3.39 | +2747, +0.22 | −7454, +4.03 | +1872, +0.76 | −3868 | +1518 |
| MACE-MATPES-r2SCAN-0 | +1789, +0.24 | +3331, −0.63 | −198, +0.95 | +3569, −1.21 | +162 | +3790 |
| GRACE-2L-OMAT | −2760, +1.66 | +421, +0.12 | −3442, +2.67 | +227, +0.16 | +439 | +1382 |

In experiment Co-Ni is a continuous FCC solid solution below the solidus. The
liquidus and solidus lie close together between the melting points of Co
(1768 K) and Ni (1728 K). On the Co side HCP is stable at low temperature
(Nishizawa and Ishida, Bull. Alloy Phase Diagr. 4, 390 (1983)).

All four potentials give the continuous FCC_A1 solution and the HCP_A3 field on
the Co side. The HCP_A3 field comes mostly from the SGTE energies of pure Co. With
MACE-MATPES-r2SCAN-0, FCC_A1 also splits into two phases below about 370 K.
The potentials differ near the melting points.

- With GRACE-2L-OMAT the liquidus and the solidus fall from Co to Ni, as in
  experiment. Any maximum or minimum is within the 2 K step of the grid.
- With the three MACE potentials the solid and the liquid meet at a shallow
  minimum, 30 to 65 K below the melting point of Ni.

The FCC_A1 L0 terms have a positive b, so the excess vibrational entropy of the
solid is negative. It makes the solid less stable at high temperature.
Without it, MACE-OMAT-0-medium, MACE-MATPES-PBE-0 and GRACE-2L-OMAT gave a
maximum up to about 80 K above the melting point of Co.

The table gives the temperature of the minimum in K, read from the
`plot_melting` grid, with steps of 0.02 in composition and 2 K in
temperature. The last column is the mean and standard deviation over ten
liquid MD runs with different random seeds, which section 4 describes.

| Potential | Our fit | Ten runs |
|---|---|---|
| MACE-OMAT-0-medium | 1698 at 40 at% Ni | 1692 ± 14 |
| MACE-MATPES-PBE-0 | 1664 at 40 at% Ni | 1650 ± 11 |
| MACE-MATPES-r2SCAN-0 | 1678 at 64 at% Ni | 1688 ± 12 |

## 3. Cr-V

Cr and V are both stable in BCC_A2. BCC_A2 is not in the default lattices, so
we add it to `lattices`. Its default terms are L0 and L1, like those of the
other default lattices. The liquid is melted at 3500 K and run at 2400 K, about
220 K above the melting points of both elements. The solids were relaxed with
the default settings.

In [ ]:
CR_V = {
    "melt_temperature": 3500,
    "liquid_temperature": 2400,
    "lattices": ["BCC_A2", "LIQUID"],
}
maker = get_maker("MACE-OMAT-0-medium", **CR_V)
flow = maker.make(["Cr", "V"])
cr_v_terms = {lattice: maker.terms[lattice] for lattice in maker.lattices}

tdbs = fit_tdbs("cr_v", ["Cr", "V"], cr_v_terms)
plot_diagrams(tdbs, ["Cr", "V"], (1000, 3000, 10), line=2043)
get_mixing_terms(tdbs, maker.lattices, 1)

![Cr-V phase diagrams with the four potentials](calphad_workflow_figures/cr_v_phase_diagrams.png)

The dotted line is the experimental minimum at 2043 K.

In our runs all BCC_A2 relaxations converged, and the largest
`relaxation_strain` was 0.006. The largest `imaginary_fraction` was 0.013,
for pure V with MACE-OMAT-0-medium. This is below the default
`max_imaginary_fraction` of 0.03. The mean squared displacement of the
liquids at 2400 K was 46 to 95 Å². The cell gives these mixing terms. The
first number is a in J/mol and the second b in J/(mol K).

| Potential | BCC_A2 L0 | BCC_A2 L1 | LIQUID L0 | LIQUID L1 |
|---|---|---|---|---|
| MACE-OMAT-0-medium | −21643, +11.23 | +958, −3.51 | −9250 | +1427 |
| MACE-MATPES-PBE-0 | −26007, +13.47 | +309, −2.99 | −6867 | −758 |
| MACE-MATPES-r2SCAN-0 | −24127, +7.44 | −2749, −2.69 | +562 | +2441 |
| GRACE-2L-OMAT | −23221, +13.80 | −1986, −2.46 | −7433 | +575 |

In experiment Cr-V is a continuous BCC solid solution. The liquidus has a
minimum at about 2043 K (1770 °C) and 30 at% V
([Liquidus Projection of the Cr-Nb-V System](https://doi.org/10.1007/s11669-026-01228-4),
which follows the assessments of Smith, Bailey and Carlson (1982) and Ghosh
(2002)).

All four potentials give the continuous BCC_A2 solution. The mixing energy of
BCC_A2 is much more negative than that of the liquid. Without the vibrational
entropy, the solid and the liquid therefore met at a maximum 300 to 700 K
above the melting points.

The excess vibrational entropy of BCC_A2 at 50 at% V is −0.22 to
−0.41 k_B/atom. Near the melting points it makes the solid less stable by 4
to 7.6 kJ/mol. With MACE-OMAT-0-medium, MACE-MATPES-PBE-0 and GRACE-2L-OMAT,
the maximum becomes a minimum at 1894 to 1976 K. This is 70 to 150 K below
the experimental minimum. But it lies at 62 to 64 at% V, not 30 at% V. With
MACE-MATPES-r2SCAN-0 the liquid L0 is positive, and a maximum remains at
2432 K and 28 at% V.

The table gives the temperature of the minimum or maximum in K, from
`pycalphad.equilibrium` on a grid with steps of 0.02 in composition and 2 K in
temperature. The last column is the mean and standard deviation over ten
liquid MD runs, as for Co-Ni.

| Potential | Our fit | Ten runs |
|---|---|---|
| MACE-OMAT-0-medium (minimum) | 1922 at 64 at% V | 1923 ± 12 |
| MACE-MATPES-PBE-0 (minimum) | 1976 at 62 at% V | 1983 ± 15 |
| MACE-MATPES-r2SCAN-0 (maximum) | 2432 at 28 at% V | 2438 ± 23 |
| GRACE-2L-OMAT (minimum) | 1894 at 64 at% V | 1910 ± 14 |

The liquid MD at 2400 K is below the maximum of MACE-MATPES-r2SCAN-0, but the
cells stayed liquid during the 20 ps run.

## 4. Choosing the liquid settings

The tests of cell size and temperature used Ni-Re.

### Size of the liquid cell

We ran each liquid SQS with three random seeds, set by `mb_velocity_seed` of
both liquid makers. These runs used 3600 K, not the 2800 K of section 1. The
next table and figure give the mean and standard deviation of L0 of the liquid
in J/mol. In the figure, open markers are the three runs. We ran GRACE-2L-OMAT
only at 864 atoms.

![L0 of the liquid against the size of the liquid cell at 3600 K](calphad_workflow_figures/liquid_l0_cell_size.png)

| Potential | 256 atoms | 864 atoms | 2048 atoms |
|---|---|---|---|
| MACE-OMAT-0-medium | −1347 ± 593 | −3381 ± 449 | −3098 ± 103 |
| MACE-MATPES-PBE-0 | −17720 ± 1897 | −17141 ± 859 | −16867 ± 951 |
| MACE-MATPES-r2SCAN-0 | −14700 ± 1149 | −15741 ± 1117 | −15250 ± 631 |
| GRACE-2L-OMAT | | −7320 ± 1736 | |

From 864 to 2048 atoms the mean changes by 270 to 490 J/mol, within the
scatter between runs. From 256 to 864 atoms it changes by 2030 J/mol with
MACE-OMAT-0-medium. So 256 atoms is too small, and the workflow uses 864
atoms by default. We did not check the size at 2800 K. On one A100 GPU a MACE
liquid run took about 15 minutes at 256 atoms and 100 minutes at 2048 atoms.

### Number of liquid runs

We ran each liquid SQS of the three binaries ten times with different random
seeds, for 20 ps at the liquid temperatures of sections 1 to 3. We also ran
each once for 100 ps. The next table gives the scatter of one 20 ps run, as
one standard deviation. The liquidus maximum or minimum is the one of Co-Ni
and Cr-V in sections 2 and 3.

| System | Liquid L0 (J/mol) | Liquidus maximum or minimum (K) | Liquid energy (meV/atom) |
|---|---|---|---|
| Ni-Re | 710 to 1200 | | 2.0 to 2.4 |
| Co-Ni | 430 to 560 | 11 to 14 | 1.3 to 1.4 |
| Cr-V | 630 to 840 | 12 to 23 | 1.3 to 1.5 |

The liquidus column leaves out GRACE-2L-OMAT for Co-Ni, which has no maximum
or minimum.

The mean of several runs scatters less, roughly with one over the square root
of their number. One run never turned a maximum into a minimum or back. The
100 ps runs showed no drift of the energy beyond this scatter, and all cells
stayed liquid. So the error is statistical.

The workflow stores the standard error of each liquid energy in
`energy_standard_error`. It comes from five blocks of the liquid MD after the
equilibration frames. In our runs it was 1.1 to 2.2 meV/atom, close to the
scatter between seeds in the table. Propagated through the fit, it gave the
scatter of L0 between runs to within about 30%. So the standard error of one
run shows whether more runs are needed.

### Temperature of the liquid MD

The liquid L0 comes from MD at `liquid_temperature`, so it depends on that
choice. The next table and figure give L0 of the liquid in J/mol at 864 atoms,
as the mean and standard deviation of three runs.

![L0 of the liquid against liquid_temperature](calphad_workflow_figures/liquid_l0_temperature.png)

| Potential | 2800 K | 3200 K | 3600 K | 4000 K |
|---|---|---|---|---|
| MACE-OMAT-0-medium | +65 ± 414 | | −3381 ± 449 | |
| MACE-MATPES-PBE-0 | −16125 ± 1706 | | −17141 ± 859 | |
| MACE-MATPES-r2SCAN-0 | −14000 ± 500 | | −15741 ± 1117 | |
| GRACE-2L-OMAT | −4693 ± 1156 | −5989 ± 652 | −7320 ± 1736 | −9137 ± 1617 |

With GRACE-2L-OMAT, L0 changes by −3.7 ± 0.8 J/mol/K. With the MACE potentials
it rises by 1.0 to 3.4 kJ/mol from 3600 to 2800 K. For PBE this rise is within
the scatter. The diagram is decided near the liquidus, which lies between 1728
and 3458 K. So `liquid_temperature` should be as low as possible while every
composition stays liquid. In all runs at 2800 K the energy stayed flat after
the first 5 ps and the mean squared displacement kept growing.

All compositions must run at the same temperature. We tried runs at 50 K above
the mean melting point of the elements of each composition, from 1778 K for Ni
to 3509 K for Re. With GRACE-2L-OMAT this gave L0 of the liquid near
−12 kJ/mol. The potential energy per atom of the liquid rises by 0.171 meV/K
for Ni and by 0.234 meV/K for Re. Over 1778 to 3509 K this difference adds
about −10 kJ/mol to L0.

## 5. Comparison with PhaseForge

Zhu et al. (npj Comput. Mater. 11, 340 (2025)) computed Ni-Re with the same
`sqs2tdb` approach in their PhaseForge code and with GRACE-2L-OMAT. Their
peritectic is at 1904 K (1631 °C), 70 K above the experimental one. We ran
PhaseForge with GRACE-2L-OMAT and its default settings and got their diagram.
Its solid mixing terms have no T part, and they agree with our a parts to
0.1 J/mol. Its L0 of the liquid is +3284 J/mol, against −3860 J/mol here.

PhaseForge runs each 32-atom liquid SQS for 2 ps in NVT, at 50 K above the
mean melting point of its elements. In our PhaseForge run the mean squared
displacement of all five cells was 0.05 to 0.11 Å². The cells stayed
crystalline, so their L0 of the liquid is a solid mixing energy. Zhu et al.
note that their agreement with experiment may come from a cancellation of
errors.

### Lattices that are not stable

Zhu et al. fit FCC_A1, BCC_A2 and HCP_A3 for every binary. Their Co-Ni diagram
has a BCC_A2 field near 50 at% Ni, which they note is not seen in experiment.
Their Cr-V diagram has an HCP_A3 field and a liquidus maximum near 3200 K.

We also relaxed the BCC_A2 SQS of Co-Ni and the FCC_A1 and HCP_A3 SQS of Cr-V
with all four potentials. These lattices are not stable for the elements.

- With three of the four potentials, the mixed BCC_A2 SQS of Co-Ni relax away
  from BCC, with a `relaxation_strain` of 0.12 to 0.17.
- The mixed FCC_A1 and HCP_A3 SQS of Cr-V relax into BCC with all four
  potentials, with a `relaxation_strain` of 0.17 to 0.41.
- The phonons of the pure elements on these lattices have imaginary modes,
  up to 5% of the modes for BCC Co and Ni and up to 26% for FCC and HCP Cr
  and V.

A fit to these energies gives the two fields of Zhu et al. The next figure
shows this for MACE-OMAT-0-medium. On the left is our fit with the lattices of
sections 2 and 3. On the right the extra lattices are added, which we do not
use. The fit uses only the energies of the extra lattices. The BCC_A2 field of Co-Ni then reaches about 2230 K. In Cr-V, HCP_A3 alone
is stable in the middle up to above 3400 K. Both fields come from SQS that are
no longer on their lattice. So we left these lattices out, and the fit job
warns when a `relaxation_strain` is above 0.1.

![Our fits and fits with the extra lattices of Zhu et al., MACE-OMAT-0-medium](calphad_workflow_figures/unstable_lattices.png)

## Known limitations

- The vibrational entropy is harmonic, from the relaxed SQS at 0 K, and the
  value at 3000 K is used at all temperatures. Anharmonic effects and thermal
  expansion are left out. The short-range order option of `sqs2tdb` is not
  used.
- The liquid L0 is the excess energy at `liquid_temperature`. The TDB file uses
  it at all temperatures. L0 changes with temperature, so the liquid has an
  excess heat capacity. Its excess entropy then cannot be zero at all
  temperatures, and the fit leaves it out. It would need free energies, for
  example from thermodynamic integration.
- The Ni side of the Ni-Re diagram is decided near 1800 K, far below 2800 K. A linear
  extrapolation of the mean GRACE-2L-OMAT L0 of three runs gives about
  −1 kJ/mol at 1800 K, against −4.7 kJ/mol at 2800 K.
- An ordered lattice takes the energies and vibrational entropies of its pure
  element end members from the force field, while FCC_A1 and HCP_A3 take them
  from SGTE. So pure Re on FCC sites can have a lower free energy in NI4MO_D1A
  than in FCC_A1. This may help the ordered phases that all four potentials
  make stable in Ni-Re.
- The mixing energies and vibrational entropies come from the force fields. A
  comparison with DFT for the same SQS would show how much of the difference
  to experiment comes from the force fields.